# Ferry Data route count outputs has similar behavior, but is not identical to, the ferry boarding data provided by insight.

In [511]:


# Cell 0: read MBTA CSV
from pathlib import Path
import pandas as pd


path = Path("MBTA_Ferry_Daily_Ridership_by_Trip%2C_Route%2C_and_Stop.csv")
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}")


df = pd.read_csv(path, low_memory=False)


print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns")
df.head()



Loaded 347,309 rows and 21 columns


,service_date,route_id,route_name,trip_freq,trip_freq_adj,sub_route,trip_id,stop_id,departure_terminal,mbta_sched_departure,...,pax_on,pax_load,pax_off,mbta_sched_arrival,actual_arrival,arrival_terminal,vessel_time_slot,trip_endpoint,travel_direction,ObjectId
0,2018/11/01 04:00:00+00,F1,F1-Hingham,M-F,All,F1-Hingham-Boston,06:00-F1-HNG-RWF,06:00-F1-HNG-RWF,Hingham,2018/11/01 10:00:00+00,...,156.0,156.0,-156,2018/11/01 10:35:00+00,2018-11-01 06:30:00,Rowes Wharf,Vessel A,Trip Endpoint,To Boston,1
1,2018/11/01 04:00:00+00,F1,F1-Hingham,M-F,All,F1-Boston-Hingham,06:50-F1-RWF-HNG,06:50-F1-RWF-HNG,Rowes Wharf,2018/11/01 10:50:00+00,...,3.0,3.0,-3,2018/11/01 11:30:00+00,2018-11-01 07:11:00,Hingham,Vessel A,Trip Endpoint,From Boston,2
2,2018/11/01 04:00:00+00,F1,F1-Hingham,M-F,All,F1-Hingham-Boston,07:45-F1-HNG-RWF,07:45-F1-HNG-RWF,Hingham,2018/11/01 11:45:00+00,...,421.0,421.0,-421,2018/11/01 12:20:00+00,2018-11-01 08:12:00,Rowes Wharf,Vessel A,Trip Endpoint,To Boston,3
3,2018/11/01 04:00:00+00,F1,F1-Hingham,M-F,All,F1-Boston-Hingham,08:30-F1-RWF-HNG,08:30-F1-RWF-HNG,Rowes Wharf,2018/11/01 12:30:00+00,...,5.0,5.0,-5,2018/11/01 13:05:00+00,2018-11-01 09:05:00,Hingham,Vessel A,Trip Endpoint,From Boston,4
4,2018/11/01 04:00:00+00,F1,F1-Hingham,M-F,All,F1-Hingham-Boston,09:15-F1-HNG-RWF,09:15-F1-HNG-RWF,Hingham,2018/11/01 13:15:00+00,...,111.0,111.0,-111,2018/11/01 13:50:00+00,2018-11-01 09:49:00,Rowes Wharf,Vessel A,Trip Endpoint,To Boston,5


In [512]:
# ensure dates are datetimes, coerce invalid to NaT
df['service_date'] = pd.to_datetime(df['service_date'], errors='coerce')

# bounds (use start <= date <= end)
# start = '2024-09-01'
# end = '2024-10-31'
start = '2024-09-01'
end = '2024-10-31'

before = len(df)
df = df[(df['service_date'] >= start) & (df['service_date'] <= end)]
after = len(df)
print(f"Rows before: {before}, after filter: {after}")

# if you want to drop rows with invalid/missing dates first:
# df = df.dropna(subset=['service_date'])

Rows before: 347309, after filter: 13138


In [513]:
# show distinct values for trip_freq (new cell)
if 'trip_freq' not in df.columns:
    print("Column 'trip_freq' not found in df")
else:
    vc = df['trip_freq'].value_counts(dropna=False)
    if vc.empty:
        print("No values found in 'trip_freq' (dataframe is empty or column has only missing values).")
    else:
        print("Value counts for 'trip_freq':")
        print(vc)
        distinct = df['trip_freq'].dropna().unique().tolist()
        distinct_sorted = sorted(distinct, key=lambda x: (str(type(x)), str(x)))
        print("\nDistinct non-null trip_freq values (sorted):")
        for v in distinct_sorted:
            print(v)

Value counts for 'trip_freq':
trip_freq
NaN        8366
M-F        3683
Sat         711
Unknown     378
Name: count, dtype: int64

Distinct non-null trip_freq values (sorted):
M-F
Sat
Unknown


In [514]:
df.to_csv("MBTA_Ferry_Ridership_Fall2024.csv", index=False)

In [515]:
df.shape[0]

13138

In [516]:
# df = df[df['trip_freq'].isin(['', 'M-F'])].copy()
# df = df[df['trip_freq'].isin(['M-F', 'Unknown'])].copy()
# df = df[df['trip_freq'].isin([None, 'M-F'])].copy()
df = df[~df['trip_freq'].isin(['Sat', 'Unknown'])].copy()
# df = df[~df['trip_freq'].isin(['Sat'])].copy()

In [517]:
df.shape[0]


12049

In [518]:
# aggregate boardings by route_id
agg_boardings = (
    df.groupby(['route_id'], as_index=False)['pax_on']
      .sum()
      .rename(columns={'pax_on': 'total_boardings'})
)

# sort for easier inspection (highest totals first)
agg_boardings = agg_boardings.sort_values(['route_id', 'total_boardings'], ascending=[True, False])


In [519]:
# agg_boardings = agg_boardings[agg_boardings['day_type_name'] == 'weekday']

In [520]:
# agg_boardings['average_boardings'] = agg_boardings['total_boardings'] / 42
agg_boardings

,route_id,total_boardings
0,F1,118575.0
1,F2H,39775.0
2,F3,16636.0
3,F4,55290.0
4,F5,4967.0
5,F6,8294.0


In [521]:
agg_boardings[['route_id', 'total_boardings']]
agg_boardings.to_csv("ferry_route_weekday_boardings_aggregate.csv", index=False)

In [522]:
# agg_boardings = agg_boardings[agg_boardings['route_id'] != "F2H"]

In [523]:
total_boardings_sum = agg_boardings['total_boardings'].sum()
print(f"Sum of total_boardings: {total_boardings_sum:,.1f}")

Sum of total_boardings: 243,537.0


In [524]:
# Scale total_boardings so that their sum matches 5,411
target_sum = 5_411

scale_factor = target_sum / total_boardings_sum

agg_boardings_scaled = agg_boardings.copy()
agg_boardings_scaled['total_boardings'] = agg_boardings_scaled['total_boardings'] * scale_factor

print(f"Scaled sum: {agg_boardings_scaled['total_boardings'].sum():,.1f}")

Scaled sum: 5,411.0


In [525]:
agg_boardings_scaled.to_csv("ferry_route_weekday_boardings_aggregate_scaled.csv", index=False)
agg_boardings_scaled

,route_id,total_boardings
0,F1,2634.545572
1,F2H,883.736455
2,F3,369.625133
3,F4,1228.454773
4,F5,110.358742
5,F6,184.279325


In [526]:
# aggregate statistics on agg_boardings.total_boardings (weekday)
tb = agg_boardings['total_boardings']

agg_stats = pd.Series({
    'count': tb.count(),
    'sum': tb.sum(),
    'mean': tb.mean(),
    'median': tb.median(),
    'std': tb.std(),
    'min': tb.min(),
    'max': tb.max()
})
print(agg_stats)

# re-aggregate by route_id (defensive) and sort
per_route = agg_boardings.groupby('route_id', as_index=False)['total_boardings'].sum()
per_route = per_route.sort_values('total_boardings', ascending=False)

# top / bottom routes
print("\nTop 10 routes by weekday total_boardings:")
print(per_route.head(10))

print("\nBottom 10 routes by weekday total_boardings:")
print(per_route.tail(10))

# add percent and cumulative share, save results
per_route['pct_share'] = per_route['total_boardings'] / per_route['total_boardings'].sum()
per_route['cum_share'] = per_route['pct_share'].cumsum()
per_route.to_csv("weekday_boardings_per_route_with_shares.csv", index=False)

# quick bar plot of top 20 routes (requires matplotlib)
try:
    import matplotlib.pyplot as plt
    per_route.set_index('route_id')['total_boardings'].head(20).plot(
        kind='bar', figsize=(10,4), title='Top 20 routes — weekday total boardings'
    )
    plt.ylabel('total_boardings')
    plt.tight_layout()
except Exception:
    pass

count          6.000000
sum       243537.000000
mean       40589.500000
median     28205.500000
std        42858.460797
min         4967.000000
max       118575.000000
dtype: float64

Top 10 routes by weekday total_boardings:
  route_id  total_boardings
0       F1         118575.0
3       F4          55290.0
1      F2H          39775.0
2       F3          16636.0
5       F6           8294.0
4       F5           4967.0

Bottom 10 routes by weekday total_boardings:
  route_id  total_boardings
0       F1         118575.0
3       F4          55290.0
1      F2H          39775.0
2       F3          16636.0
5       F6           8294.0
4       F5           4967.0


Difference from Insight's Data



In [ ]:
'''
Insight's Data

route_id        	route_long_name	        Group	    Ridership	route_gtfs  	    Mode		corresponding_route_id
Boat-EastBoston&T	East Boston Ferry	    31      	283     	Boat-EastBoston	    Ferry		F3
Boat-F1&T           Hingham/Hull Ferry  	30      	3625    	Boat-F1	            Ferry		F1
Boat-F4&T       	Charlestown Ferry     	29      	1089    	Boat-F4	            Ferry		F4
Boat-F6&T	        Winthrop/Quincy Ferry	33      	224     	Boat-F6	            Ferry		F6
Boat-Lynn&T	        Lynn Ferry          	32      	189	        Boat-Lynn	        Ferry		F5

it is worth noting that the total ridership number of 5410 is close to the 5411 defined in 
"MBTA Monthly Ridership By Mode and Line"
https://docs.google.com/spreadsheets/d/1eyboPIVq72FALLALQT5rqU1SFYZ6WFtetSiEde9gzmo/edit?gid=0#gid=0
(derived from https://mbta-massdot.opendata.arcgis.com/datasets/2048258a18354256a650d41f8fe4532c_0/explore)

This suggests that the filtering criteria used in this analysis may be derived from ferry boarding and scaled to 5411.

scaled data is as follows

Parsed MBTA Ferry Data (daily boardings) that is then scaled to 5410 (to match total ridership for Insight
- this may need to be 5411 if alignment with MBTA's monthly report is desired)
route_id	insight_total_boardings	    daily boardings     daily boardings scaled to 5410
F1	        3625	                    118575	            3148.235441
F3	        283	                        16636	            441.6955075
F4	        1089	                    55290	            1467.981763
F6	        224	                        4967	            131.8767484
F5	        189	                        8294	            220.2105397
Total	    5410	                    203762	            5410

insight's data and ferry data (scaled to 5410) does not align exactly.
Matching numbers would be ideal.
Given the small differences, this may be acceptable for current purposes.

If desire for exact match, there can be an effort to identify exact date range for ferry data for aggregation.
This seems unnecessary as the aggregation logic aligns with logics found in bus and commuter rail codes.
'''